# 4. Statistical Modelling

In this notebook, we use the observations made during the previous stages of the project in order to develop and evaluate models which can accurately determine the price of a used car from its observable characteristics. In doing this, we hope to further understand which characteristics contribute most to these predictions and how well the models generalise.

In [1]:
import pandas as pd
import numpy as np
import joblib

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from src.hierarchical_imputer import HierarchicalImputer

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split, KFold, GroupKFold, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

from sklearn.inspection import permutation_importance

In [2]:
original_df = pd.read_parquet("../data/car_data.parquet")
original_df.head()

,Make,Model,Variant,CarPrice,Year,Reg,BodyType,Miles,EngineVol,Transmission,FuelType,NumOwner,ULEZ,BrandNew,EngineBHP
0,AC,Cobra,NaN,89995.0,2001,X,convertible,14400.0,4.9,manual,petrol,5,False,False,225.0
1,AC,Cobra,NaN,92500.0,2019,T,convertible,650.0,NaN,manual,petrol,<NA>,False,False,NaN
2,AC,Cobra,NaN,109995.0,2000,X,convertible,21600.0,3.5,manual,petrol,3,False,False,NaN
3,AC,Cobra,NaN,124950.0,1989,F,convertible,2750.0,NaN,manual,petrol,<NA>,False,False,NaN
4,AC,Cobra,NaN,124950.0,1989,E,convertible,15142.0,5.0,manual,petrol,<NA>,False,False,NaN


We aim to predict the column `CarPrice`. Analysis has shown that the `Year` column provides the most amount of predictive information because many of the other columns have overlapping information with it. In addition, only ~48k of records do not feature a year value. Therefore, we make the decision to remove any entry from the dataset which does not feature a year column.

Furthermore, we have discovered that the `Variant` column is unreliable and the `Reg` column is too dependent on `Year`. Therefore, we also remove these columns from the modelling dataset.

In [3]:
df = original_df.dropna(subset=["CarPrice", "Year"])
df = df.drop(columns=["Variant", "Reg"])

df.head(20)

,Make,Model,CarPrice,Year,BodyType,Miles,EngineVol,Transmission,FuelType,NumOwner,ULEZ,BrandNew,EngineBHP
0,AC,Cobra,89995.0,2001,convertible,14400.0,4.9,manual,petrol,5,False,False,225.0000
1,AC,Cobra,92500.0,2019,convertible,650.0,NaN,manual,petrol,<NA>,False,False,NaN
2,AC,Cobra,109995.0,2000,convertible,21600.0,3.5,manual,petrol,3,False,False,NaN
3,AC,Cobra,124950.0,1989,convertible,2750.0,NaN,manual,petrol,<NA>,False,False,NaN
4,AC,Cobra,124950.0,1989,convertible,15142.0,5.0,manual,petrol,<NA>,False,False,NaN
5,AC,Cobra,145500.0,2022,convertible,500.0,NaN,manual,petrol,<NA>,False,False,NaN
6,AC,Cobra,124950.0,1989,convertible,2750.0,NaN,manual,petrol,<NA>,False,False,NaN
7,AC,Cobra,89995.0,2001,convertible,14400.0,4.9,manual,petrol,5,False,False,225.0000
8,Abarth,124 Spider,24275.0,2019,convertible,10313.0,1.4,automatic,petrol,2,True,False,167.0000
9,Abarth,124 Spider,24275.0,2019,convertible,10313.0,1.4,automatic,petrol,2,True,False,167.0000


In [4]:
df.to_parquet("../data/modelling_dataset.parquet")

## 4.1. Data Preprocessing

Before constructing the pre-processing pipeline, we ensure all strings in the dataset are lowercase and convert boolean columns to 0/1 binary columns.

In [5]:
for col in df.select_dtypes(str).columns:
    df[col] = df[col].str.lower()

In [6]:
for col in df.select_dtypes(bool).columns:
    df[col] = df[col].astype(int)

In [7]:
df.head(20)

,Make,Model,CarPrice,Year,BodyType,Miles,EngineVol,Transmission,FuelType,NumOwner,ULEZ,BrandNew,EngineBHP
0,ac,cobra,89995.0,2001,convertible,14400.0,4.9,manual,petrol,5,0,0,225.0000
1,ac,cobra,92500.0,2019,convertible,650.0,NaN,manual,petrol,<NA>,0,0,NaN
2,ac,cobra,109995.0,2000,convertible,21600.0,3.5,manual,petrol,3,0,0,NaN
3,ac,cobra,124950.0,1989,convertible,2750.0,NaN,manual,petrol,<NA>,0,0,NaN
4,ac,cobra,124950.0,1989,convertible,15142.0,5.0,manual,petrol,<NA>,0,0,NaN
5,ac,cobra,145500.0,2022,convertible,500.0,NaN,manual,petrol,<NA>,0,0,NaN
6,ac,cobra,124950.0,1989,convertible,2750.0,NaN,manual,petrol,<NA>,0,0,NaN
7,ac,cobra,89995.0,2001,convertible,14400.0,4.9,manual,petrol,5,0,0,225.0000
8,abarth,124 spider,24275.0,2019,convertible,10313.0,1.4,automatic,petrol,2,1,0,167.0000
9,abarth,124 spider,24275.0,2019,convertible,10313.0,1.4,automatic,petrol,2,1,0,167.0000


#### Data Splitting

We choose a 80/20 split between training and test data.

In [8]:
X = df.drop(columns="CarPrice")
y = df["CarPrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

#### Categorical Pre-processing

There are a few categorical columns in the dataset:

- `Make`
- `Model`
- `BodyType`
- `Transmission`
- `FuelType`

We use one-hot encoding with these columns.

Every single entry in the dataset has a `Make` and `Model` entry, so we do not need to concern ourselves with imputation for these columns. However, the other columns have several missing entries. However, these are very likely to be tied to the make/model of the car. Therefore, we follow a hierarchical imputation strategy, for any missing data we:

- First we try and fill the gap with the modal category given the (`Make`, `Model`, `Year`) tuple
- Then we try and fill the gap with the modal category given the (`Make`, `Model`) pairing
- Then we try and fill the gap using the modal category given just the `Make`
- If neither of these work, we just take the global modal category

We have defined a custom `sklearn` transformer which does exactly this job, see `src/hierarchical_imputer.py`. In terms of the `sklearn` pipeline, since we are using `Year` which is not categorical, this imputation step will take place **before** the "categorical preprocessing pipeline".

In [9]:
categorical_preprocessing = Pipeline([
    (
        "one_hot_encoder",
        OneHotEncoder(handle_unknown="ignore")
    )
])

categorical_preprocessing

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('one_hot_encoder', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to 

#### Numerical Pre-processing

Most of the remaining columns are numerical:

- `Year`
- `Miles`
- `EngineVol`
- `NumOwner`
- `EngineBHP`

EDA showed that a missing `NumOwner` is potentially an indicator of higher prices.
`EngineBHP` and `EngineVol` are likely similar to `Transmission` and `FuelType` in the sense that most cars of the same model will have the same engine volume and BHP.
EDA showed that `Miles` and `Year` were negatively correlated.
Thus we also use our `HierarchicalImputer` with these, except with numerical data we will impute the _median_.

So, `EngineBHP` and `EngineVol` will use the same imputation hierarchy as the categorical features. However, since `Miles` and `NumOwner` are most likely independent of `Make` and `Model`, we only impute based on the median given the `Year`. `Year` does not require imputation; recall earlier that we removed all rows from the dataset which did not have a year entry.

Again, imputation happens before the "numerical preprocessing pipeline" - which consists of nothing.

In [10]:
numerical_preprocessing = "passthrough"

#### Boolean Pre-processing

The remaining columns (`ULEZ` and `BrandNew`) are boolean/binary columns which we input to the model as 0/1 integers. We do not need to handle missing values since we previously made the assumption that NaN = False. Thus, the "boolean preprocessing pipeline" consists of nothing.

In [11]:
boolean_preprocessing = "passthrough"

#### Combining Pipelines

We now construct the whole preprocessing pipeline. As stated earlier, we do all imputation prior to the data-type specific pipelines.

In [12]:
categorical_features = [
    "Make",
    "Model",
    "BodyType",
    "Transmission",
    "FuelType",
]

numerical_features = [
    "Year",
    "Miles",
    "EngineVol",
    "EngineBHP",
    "NumOwner"
]

boolean_features = [
    "ULEZ",
    "BrandNew"
]

std_preprocessing = Pipeline([
    (
        "make_model_year_hierarchical_imputer_modal",
        HierarchicalImputer(
            [
                "FuelType",
                "Transmission",
                "BodyType"
            ],
            [
                ["Make", "Model", "Year"],
                ["Make", "Model"],
                ["Make"]
            ],
            method="mode"
        )
    ),
    (
        "make_model_year_hierarchical_imputer_median",
        HierarchicalImputer(
            [
                "EngineVol",
                "EngineBHP"
            ],
            [
                ["Make", "Model", "Year"],
                ["Make", "Model"],
                ["Make"]
            ],
            method="median"
        )
    ),
    (
        "year_based_hierarchical_imputer_median",
        HierarchicalImputer(
            [
                "Miles",
                "NumOwner"
            ],
            [
                ["Year"]
            ],
            method="median",
            mark_missing=True
        )
    ),

    (
        "column_transformer",
        ColumnTransformer([
            ("numerical", numerical_preprocessing, numerical_features),
            ("categorical", categorical_preprocessing, categorical_features),
            ("boolean", boolean_preprocessing, boolean_features)
        ])
    )
])

std_preprocessing

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('make_model_year_hierarchical_imputer_modal', ...), ('make_model_year_hierarchical_imputer_median', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,columns,"['FuelType', 'Transmission', ...]"
,hierarchy,"[['Make', 'Model', ...], ['Make', 'Model'], ...]"
,method,'mode'
,mark_missing,False
,columns,"['EngineVol', 'EngineBHP']"
,hierarchy,"[['Make', 'Model', ...], ['Make', 'Model'], ...]"
,method,'median'


## 4.3. Model Development

We develop models of this dataset using 4 different families:

1. Linear Regression
2. Random Forest
3. Gradient Boosting
4. Neural Networks

Each of these models will be developed independently and follow the same process:

1. Identify potential model variants (e.g. with and without standardisation)
2. Compare metrics achieved by 5-fold cross-validation
3. Choose a single variant to propose as the main model from this family
4. Assess the effect of repeated data on the model using grouped 3-fold grouped cross-validation
5. Assess generalisation by testing on the test set

In cross-validation and testing, we measure MAE, RMSE and $R^2$ score. Therefore, by the end of this section we should have collected 3 sets of each metric for each model: one set from initial CV, one set from grouped CV and one set from testing.

The reason why we assess with grouped $k$-fold cross-validation is because the dataset was originally a list of observed used car sales listings. Throughout the project, we have stripped away many of the things that made each record unique (e.g. `CarAttentionGrabber`, etc.). Therefore, we now have many rows which share the same:

- `Make`
- `Model`
- `Year`
- `Miles`
- `EngineBHP`
- `EngineVol`

In [13]:
# Helper methods for cross-validation and testing

# Calculate groups for Grouped K-fold CV
group_columns = [
    "Make",
    "Model",
    "Year",
    "Miles",
    "EngineBHP",
    "EngineVol"
]

cv_groups = pd.factorize(
    pd.MultiIndex.from_frame(X_train[group_columns])
)[0]

# Test a model on the testing set and return dataframe of metrics
def test_model(model, model_name="Model"):
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(
        mean_squared_error(y_test, y_pred)
    )
    r2 = r2_score(y_test, y_pred)

    return pd.DataFrame({
        model_name: {"MAE": mae, "RMSE": rmse, "R2": r2}
    }).T


# Cross-validation of multiple models using n_folds discussed above
# Return a dataframe of metrics (mean and std of observed metrics) as well as all observations
def cross_validate_models(model_dict, is_grouped=False, n_jobs=-1):
    cv = None
    groups=None
    if is_grouped:
        cv = GroupKFold(
            n_splits=3,
            shuffle=True,
            random_state=42
        )
        groups=cv_groups
    else:
        cv = KFold(
            n_splits=5,
            shuffle=True,
            random_state=42
        )
    
    metrics = {}
    all_cv_data = {}

    for n, m in model_dict.items():
        cv_data = cross_validate(
            m,
            X_train,
            y_train,
            cv=cv,
            groups=groups,
            scoring={
                "MAE": "neg_mean_absolute_error",
                "RMSE": "neg_root_mean_squared_error",
                "R2": "r2"
            },
            n_jobs=n_jobs
        )

        mae = -cv_data["test_MAE"]
        rmse = -cv_data["test_RMSE"]
        r2 = cv_data["test_R2"]

        metrics[n] = {
            "MAE_Mean": mae.mean(),
            "RMSE_Mean": rmse.mean(),
            "R2_Mean": r2.mean(),
            "MAE_SD": mae.std(),
            "RMSE_SD": rmse.std(),
            "R2_SD": r2.std()
        }

        all_cv_data[n] = cv_data
    
    return pd.DataFrame(metrics).T, all_cv_data


### 4.3.1. Baseline Model

Our baseline model will be the model which always predicts the median price. This is easily implementable using `sklearn`'s `DummyRegressor`. Since we are making a constant prediction, the preprocessing pipeline created in the previous section is not needed.

In [14]:
baseline_model = DummyRegressor(strategy="median")

In [15]:
baseline_cv_metrics, _ = cross_validate_models({"Baseline": baseline_model})
baseline_cv_metrics

,MAE_Mean,RMSE_Mean,R2_Mean,MAE_SD,RMSE_SD,R2_SD
Baseline,10807.132327,22464.410625,-0.046392,31.023583,1020.677465,0.004426


In [16]:
baseline_cv_grouped_metrics, _ = cross_validate_models({"Baseline": baseline_model}, is_grouped=True)
baseline_cv_grouped_metrics

,MAE_Mean,RMSE_Mean,R2_Mean,MAE_SD,RMSE_SD,R2_SD
Baseline,10807.007892,22473.884399,-0.045987,34.995538,748.399351,0.002859


In [17]:
baseline_model.fit(X_train, y_train)

,"strategy strategy: {""mean"", ""median"", ""quantile"", ""constant""}, default=""mean""Strategy to use to generate predictions.* ""mean"": always predicts the mean of the training set* ""median"": always predicts the median of the training set* ""quantile"": always predicts a specified quantile of the training set, provided with the quantile parameter.* ""constant"": always predicts a constant value that is provided by the user.",'median'
,"constant constant: int or float or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
,"quantile quantile: float in [0.0, 1.0], default=NoneThe quantile to predict using the ""quantile"" strategy. A quantile of0.5 corresponds to the median, while 0.0 to the minimum and 1.0 to themaximum.",None
Name,Type,Value
"constant_ constant_: ndarray of shape (1, n_outputs)Mean or median or quantile of the training targets or constant valuegiven by the user.","ndarray[float64](1, 1)",[[14449.]]
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X` hasfeature names that are all strings.","ndarray[object](12,)","['Make','Model','Year',...,'ULEZ','BrandNew','EngineBHP']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,12
n_outputs_ n_outputs_: intNumber of outputs.,int,1


In [18]:
baseline_errors = test_model(baseline_model, model_name="Baseline Model")
baseline_errors

,MAE,RMSE,R2
Baseline Model,10806.906708,23061.744636,-0.043301


Our baseline model has an mean absolute error of £10,806,91. This means that on average, it predicts with an error of £10,806. 
It has a RMSE of £23,061.74. This indicates that some predictions made by the model are very far from the actual price. This is plausable considering that the dataset includes some very luxury cars, some reaching £2mil in price.
The $R^2$ score is -0.0433. This means that the model has a worse MSE than the constant prediction of the mean price. This is to be expected again since the mean is the best possible constant prediction for MSE, and we are constantly selecting the median which is different from the mean due to right-skew (demonstrated by EDA).

### 4.3.2. Linear Regression

Our first real model is a linear regression model. We create two pipelines, one which ends the preprocessing of numerical columns with standard scaling, and another which leaves the columns unscaled. We compare these pipelines using cross-evaluation on the resulting regression models. We then select the optimal preprocessing configuration and test generalisation using the test set.

In [19]:
# Create the preprocessing pipeline which scales numerical columns
scaled_preprocessing = clone(std_preprocessing).set_params(column_transformer__numerical=StandardScaler())
scaled_preprocessing

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('make_model_year_hierarchical_imputer_modal', ...), ('make_model_year_hierarchical_imputer_median', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,columns,"['FuelType', 'Transmission', ...]"
,hierarchy,"[['Make', 'Model', ...], ['Make', 'Model'], ...]"
,method,'mode'
,mark_missing,False
,columns,"['EngineVol', 'EngineBHP']"
,hierarchy,"[['Make', 'Model', ...], ['Make', 'Model'], ...]"
,method,'median'


In [20]:
scaled_linear_regression_model = Pipeline([
    ("preprocessor", scaled_preprocessing),
    ("model", LinearRegression())
])

unscaled_linear_regression_model = Pipeline([
    ("preprocessor", std_preprocessing),
    ("model", LinearRegression())
])

In [21]:
lr_cv_metrics, _ = cross_validate_models({
    "UnscaledLR": unscaled_linear_regression_model,
    "ScaledLR": scaled_linear_regression_model
})

lr_cv_metrics

,MAE_Mean,RMSE_Mean,R2_Mean,MAE_SD,RMSE_SD,R2_SD
UnscaledLR,5937.074944,14374.286935,0.572611,48.621488,1438.873671,0.043936
ScaledLR,3575.461536,9595.301929,0.805282,29.577599,2155.388188,0.071121


Cross-validation has indicated that standardisation improves the predictive performance of linear regression substantially with respect to all metrics. It achieved:

- Lower MAE: £3,575 vs £5,937 (~40% decrease)
- Lower RMSE: £9,595 vs £14,374 (~33% decrease)
- Higher $R^2$: 0.805 vs 0.573 (~41% increase)

Therefore scaling appears to significantly improve the predictive performance of the Linear Regression model. So we choose this model for testing.

In [22]:
linear_regression_model = scaled_linear_regression_model

lr_cv_grouped_metrics, _ = cross_validate_models({
    "ScaledLR": linear_regression_model
})

lr_cv_grouped_metrics

,MAE_Mean,RMSE_Mean,R2_Mean,MAE_SD,RMSE_SD,R2_SD
ScaledLR,3575.461536,9595.301929,0.805282,29.577599,2155.388188,0.071121


In [23]:
linear_regression_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('make_model_year_hierarchical_imputer_modal', ...), ('make_model_year_hierarchical_imputer_median', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,columns,"['FuelType', 'Transmission', ...]"
,hierarchy,"[['Make', 'Model', ...], ['Make', 'Model'], ...]"
,method,'mode'


In [24]:
linear_regression_errors = test_model(linear_regression_model, model_name="Linear Regression")
linear_regression_errors

,MAE,RMSE,R2
Linear Regression,3555.510455,9553.969929,0.820942


On the testing set, the (Scaled) Linear Regression Model achieved an MAE of £3,556, a RMSE of £9,554 and an $R^2$ score of 0.8209. These are closely aligned to the results achieved from cross-validation and thus indicate that the model generalised well to the full dataset.

### 4.3.3. Random Forest

Next, we try a Random Forest regression model. This is a good advancement from Linear Regression because decision trees are non-linear transformations and Random Forests are simply an ensemble of decision trees. Therefore, they will allow us to capture the many non-linear relationships represented in the dataset.

Random Forests are invariant under standardisation (or any other monotone transformations for that matter) and thus we do **not** scale our numerical features. Thus our preprocessing pipeline is exactly the one we produced in section 4.1.

#### Constrained Random Forest

As a quick proof of concept because unrestricted forest models can be computationally expensive, we gather some metrics via 3-fold cross validation for a restricted random forest model with 50 trees, 20 depth and at least 5 observations per leaf.

In [25]:
constrained_model = Pipeline([
    ("preprocessor", std_preprocessing),
    ("model", RandomForestRegressor(
        n_estimators=50,
        max_depth=20,
        min_samples_leaf=5,

        random_state=42,
        n_jobs=-1
    ))
])

In [26]:
constrained_metrics, _ = cross_validate_models({"ConstrainedRF": constrained_model}, n_jobs=1)
constrained_metrics

,MAE_Mean,RMSE_Mean,R2_Mean,MAE_SD,RMSE_SD,R2_SD
ConstrainedRF,1586.169428,7940.003257,0.866399,16.238233,2059.659594,0.057736


The constrainted Random Forest model constitutes a substantial improvement upon the performance of the (scaled) Linear Regression model using metrics gathered by 5-fold cross-validation. A greater computational load could potentially yield an even better model.

#### Unconstrained Random Forest

Since the constrained Random Forest model indicated substantial improvements over our previous models, we also verify an unconstrained random forest. We keep 50 trees but we do not constraint the depth nor the minimum number of evaluations per leaf.

In [27]:
unconstrained_model = Pipeline([
    ("preprocessor", std_preprocessing),
    ("model", RandomForestRegressor(
        n_estimators=50,
        max_depth=None,
        min_samples_leaf=1,
        
        random_state=42,
        n_jobs=-1
    ))
])

In [28]:
unconstrained_metrics, data = cross_validate_models({"UnconstrainedRF": unconstrained_model}, n_jobs=1)

rf_cv_metrics = pd.concat([constrained_metrics, unconstrained_metrics], ignore_index=False)
rf_cv_metrics

,MAE_Mean,RMSE_Mean,R2_Mean,MAE_SD,RMSE_SD,R2_SD
ConstrainedRF,1586.169428,7940.003257,0.866399,16.238233,2059.659594,0.057736
UnconstrainedRF,1006.360090,6629.265674,0.903056,19.539373,2375.958128,0.060829


The unconstrained model gives rise to a substantial improvement in the Random Forest methodology, we see:

- 37% decrease in MAE (£1,006 vs. £1,586)
- 17% decrease in RMSE (£6,629 vs. £7,940)
- 4% increase in $R^2$ score (0.903 vs. 0.866)

#### Further Examination of Unconstrained Model

5-fold cross-validation indicated that the unconstrained forest model makes the most accurate price predictions. Thus, we choose it as our main Random Forest model. However, the immense accuracy demonstrated by cross-validation may be an indication of some form of information leakage. We suspect that the RF model may be benefitting from the repeated entries. This is precisely the purpose of our additional grouped cross-validation check.

In [29]:
random_forest_model = unconstrained_model
rf_cv_grouped_metrics, _ = cross_validate_models({"UnconstrainedRF": random_forest_model}, is_grouped=True, n_jobs=1)

rf_cv_grouped_metrics

,MAE_Mean,RMSE_Mean,R2_Mean,MAE_SD,RMSE_SD,R2_SD
UnconstrainedRF,1599.96519,5972.839126,0.922826,10.855622,1701.440954,0.039004


The metrics by the unconstrained random forest model in grouped 3-fold cross-validation is extremely similar to those gained in ungrouped 5-fold cross-validation. In fact, it has demonstrated a slightly better performance with respect to $R^2$ and RMSE with:

- RMSE of £5,973 vs £6,629
- $R^2$ score of 0.923 vs 0.903

Of course, the difference in the number of folds mean that we must be careful in comparing these scores directly, and we shouldn't treat this as a rigorous comparison. However, if the model's performance were inflated by the repeated entries, we would have seen a substantial decrease in performance. 

In [30]:
random_forest_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('make_model_year_hierarchical_imputer_modal', ...), ('make_model_year_hierarchical_imputer_median', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,columns,"['FuelType', 'Transmission', ...]"
,hierarchy,"[['Make', 'Model', ...], ['Make', 'Model'], ...]"
,method,'mode'


In [31]:
random_forest_errors = test_model(random_forest_model, "Random Forest")
random_forest_errors

,MAE,RMSE,R2
Random Forest,901.976137,6199.094184,0.924616


After refitting on the test set, the random forest model demonstrated a similar performance as in 5-fold cross-validation with:

- MAE: £902
- RMSE: £6,199
- $R^2$ score: 0.925

### 4.3.4. Gradient Boosting

Thirdly, we try an algorithm from the gradient boosting family, another family of tree-based ensemble methods. In random forest models, we train $n$ decision trees independently on $n$ bootstrap samples of training data; meaning that trees can be constructed in parallel. In gradient boosting models, we train $n$ decision trees sequentially such that tree $k$ is fitted to the residuals remaining after tree $k-1$ (and tree $1$ is fitted to the initial residuals); each tree is constructed in order to reduce the error remaining in the current ensemble.

Since this family takes a different approach to training a tree-based ensemble, it is not necessarily guaranteed to perform better or worse than a Random Forest model. We must establish empirically whether an independent or sequential learning strategy is better for this dataset.

Akin to the Random Forest subsection, we propose two models and compare them using cross-validation. However, our "unconstrained" equivalent will not be completely unconstrained like in Random Forests. This is because the sequential approach means that it is more beneficial to have each individual tree being fairly weak (i.e. shallow). For this reason, we will have 100 trees in our Gradient Boosting models, rather than 50 like we did in our Random Forest models.

#### Constrained Gradient Boosting

Each tree is more shallow (max depth of 3) and allow more observation in leaves (minimum of 5). This serves primarily as a computational experiment.

In [32]:
constrained_model = Pipeline([
    ("preprocessor", std_preprocessing),
    ("model", GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        min_samples_leaf=5,
        random_state=42
    ))
])

In [33]:
constrained_metrics, _ = cross_validate_models({"ConstrainedGB": constrained_model}, n_jobs=1)
constrained_metrics

,MAE_Mean,RMSE_Mean,R2_Mean,MAE_SD,RMSE_SD,R2_SD
ConstrainedGB,3107.383375,9551.826261,0.809404,12.681628,1919.324051,0.059077


#### Flexible Gradient Boosting

This model allows for deeper trees (max depth of 5) and only allows 1 node per leaf. This allows for a more flexible expression of the dataset.

In [34]:
flexible_model = Pipeline([
    ("preprocessor", std_preprocessing),
    ("model", GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        min_samples_leaf=1,
        random_state=42
    ))
])

In [35]:
flexible_metrics, _ = cross_validate_models({"FlexibleGB": flexible_model})

flexible_metrics

,MAE_Mean,RMSE_Mean,R2_Mean,MAE_SD,RMSE_SD,R2_SD
FlexibleGB,2580.345963,7634.701655,0.875639,19.371451,2081.037469,0.057378


The "flexible" model provided a substantial improvement over the "constrained" model in cross-validation. We saw:

- 17% reduction in MAE (£2,580 vs. £3,107)
- 20% reduction in RMSE (£7,635 vs. £9,552)
- 8% increase in $R^2$ (0.809 vs. 0.876)

The flexible model certainly outperformed the constrained Gradient Boosting model. However, it's not quite as accurate as we would hope, especially when we consider the accuracy of the Random Forest models. Of course, we did state earlier that it's wrong to expect it to be better or worse, but our flexible model is not very large and we suspect this may be related to the number of trees. So for good measure, we try doubling the amount of trees.

#### Larger Flexible Gradient Boosting

We use the same parameters as the previous model, except we allow for 200 trees instead of 100. We also half the learning rate.

In [36]:
larger_flexible_model = Pipeline([
    ("preprocessor", std_preprocessing),
    ("model", GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=5,
        min_samples_leaf=1,
        random_state=42
    ))
])

In [37]:
larger_flexible_metrics, _ = cross_validate_models({"LargerFlexibleGB": larger_flexible_model})

gb_cv_metrics = pd.concat([constrained_metrics, flexible_metrics, larger_flexible_metrics], ignore_index=False)
gb_cv_metrics

,MAE_Mean,RMSE_Mean,R2_Mean,MAE_SD,RMSE_SD,R2_SD
ConstrainedGB,3107.383375,9551.826261,0.809404,12.681628,1919.324051,0.059077
FlexibleGB,2580.345963,7634.701655,0.875639,19.371451,2081.037469,0.057378
LargerFlexibleGB,2554.828740,7619.334906,0.876042,17.230669,2103.297734,0.058004


The difference between the larger and standard sized flexible models is less substantial than that between constrained and flexible:

- 1% reduction in MAE (£2,555 vs. £2,580)
- 0.2% reduction in RMSE (£7,619 vs. £7,634)
- 0.04% increase in $R^2$ (0.8760 vs 0.8756)

Doubling the size of the model has given a negligible return in terms of metrics in cross-validation.

#### Further Examination of the Flexible Model

We select the standard-sized flexible model as our main Gradient Boosting model. Whilst the larger flexible model provided better metrics numerically under cross-validation, this improvement was negligble compared to the extra computational work required for prediction. Just as in the previous subsections, we also gather metrics based on grouped 3-fold cross-validation and also the general performance on the testing data.

In [38]:
gradient_boosting_model = flexible_model
gb_cv_grouped_metrics, _ = cross_validate_models({"FlexibleGB": gradient_boosting_model}, is_grouped=True)

gb_cv_grouped_metrics

,MAE_Mean,RMSE_Mean,R2_Mean,MAE_SD,RMSE_SD,R2_SD
FlexibleGB,2566.111966,6644.834988,0.907453,1.933364,1220.027521,0.028254


In [39]:
gradient_boosting_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('make_model_year_hierarchical_imputer_modal', ...), ('make_model_year_hierarchical_imputer_median', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,columns,"['FuelType', 'Transmission', ...]"
,hierarchy,"[['Make', 'Model', ...], ['Make', 'Model'], ...]"
,method,'mode'


In [40]:
gradient_boosting_errors = test_model(gradient_boosting_model, "Gradient Boosting")
gradient_boosting_errors

,MAE,RMSE,R2
Gradient Boosting,2589.207113,7760.978134,0.881843


The model's performance on the testing set is consistent with its performance during cross-validation:

- MAE: £2,859
- RMSE: £7,761
- $R^2$: 0.882

This indicates good generalisation to the dataset.

### 4.3.5. Neural Network

The previous two methods were both tree-based ensemble method which used hierarchical decisions in order to partition the feature space. Neural Networks offer a completely different modelling approach as they function as function approximators rather than space partitioners with their flexibility motivated by the Universal Approximation Theorem. The theorem states that neural networks satisfying certain conditions are able to approximate continuous functions to arbitrary accuracy. This suggests that they may be capable of approximating the non-linear relationships between our features and car price.

All of the models produced in this subsection will make use of the standardised (scaled) preprocessing pipeline created during our Linear Regression experiments and will be basic multi-layer perceptrons.

#### Small Neural Network

Our initial models will investigate the effect of network size, we train a "small" model and a "large" model and both will be trained for at most 250 epochs (with allowed early stopping). This first, smaller model will have two layers of size 64 and 32 respectively.

In [41]:
small_model = Pipeline([
    ("preprocessor", scaled_preprocessing),
    ("model", MLPRegressor(
        hidden_layer_sizes=(64, 32),
        max_iter=250,
        early_stopping=True,
        random_state=42
    ))
])

#### Larger Neural Network

The "large" model will have two layers of size 128 and 64 respectively.

In [42]:
large_model = Pipeline([
    ("preprocessor", scaled_preprocessing),
    ("model", MLPRegressor(
        hidden_layer_sizes=(128, 64),
        max_iter=250,
        early_stopping=True,
        random_state=42
    ))
])

In [43]:
size_cv_metrics, _ = cross_validate_models({
    "SmallNN": small_model,
    "LargeNN": large_model
})

size_cv_metrics

/home/jenson/Documents/data-science-proj-2/pyenv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (250) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/jenson/Documents/data-science-proj-2/pyenv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (250) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/jenson/Documents/data-science-proj-2/pyenv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (250) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/jenson/Documents/data-science-proj-2/pyenv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (250) 

,MAE_Mean,RMSE_Mean,R2_Mean,MAE_SD,RMSE_SD,R2_SD
SmallNN,1521.450384,6432.138337,0.910482,22.433324,1824.828888,0.042592
LargeNN,1516.696852,7077.408568,0.892456,80.059971,1892.988084,0.044680


The smaller network achieved better cross-validation performance than the larger network:

- MAE: £1,521 vs £1,517
- RMSE: £6,432 vs £7,077
- $R^2$: 0.91 vs 0.892

However, on some cross-validation trials, the smaller network did not converge in the 250 epochs.

#### Prolonged-Training Neural Network

We test to see if increasing the amount of training epochs would further improve the performance of the smaller neural network model on the dataset.

In [44]:
prolonged_small_model = small_model = Pipeline([
    ("preprocessor", scaled_preprocessing),
    ("model", MLPRegressor(
        hidden_layer_sizes=(64, 32),
        max_iter=500,
        early_stopping=True,
        random_state=42
    ))
])

In [45]:
prolonged_cv_metrics, _ = cross_validate_models({
    "ProlongedSmallNN": prolonged_small_model
})

nn_cv_metrics = pd.concat([size_cv_metrics, prolonged_cv_metrics])
nn_cv_metrics

,MAE_Mean,RMSE_Mean,R2_Mean,MAE_SD,RMSE_SD,R2_SD
SmallNN,1521.450384,6432.138337,0.910482,22.433324,1824.828888,0.042592
LargeNN,1516.696852,7077.408568,0.892456,80.059971,1892.988084,0.044680
ProlongedSmallNN,1515.082384,6404.730222,0.911188,25.757394,1831.318782,0.042493


Increasing the maximum number of training epochs improved performance under cross-validation but only marginally:

- MAE: £1,515 vs. £1,521
- RMSE: £6,405 vs. £6,432
- $R^2$: 0.911 vs. 0.910

Since the models both allowed for early stopping, it could potentially be that the small model only required, say, 10 extra epochs to converge rather than the 250 we added in the prolonged model. Nonetheless, we select the small model with 250 epochs as our main neural network model.

#### Further Examination of the Small Neural Network

We gather metrics of small model under grouped 3-fold cross-validation, as well as testing its generalisation to the testing set.

In [46]:
neural_network_model = small_model
nn_cv_grouped_metrics, _ = cross_validate_models({"SmallNN": neural_network_model}, is_grouped=True)

nn_cv_grouped_metrics

,MAE_Mean,RMSE_Mean,R2_Mean,MAE_SD,RMSE_SD,R2_SD
SmallNN,1668.356137,7303.829316,0.886344,68.53394,1700.658416,0.041941


In [47]:
neural_network_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('make_model_year_hierarchical_imputer_modal', ...), ('make_model_year_hierarchical_imputer_median', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,columns,"['FuelType', 'Transmission', ...]"
,hierarchy,"[['Make', 'Model', ...], ['Make', 'Model'], ...]"
,method,'mode'


In [48]:
neural_network_errors = test_model(neural_network_model, "Neural Network")
neural_network_errors

,MAE,RMSE,R2
Neural Network,1499.36919,6835.787653,0.908335


Testing performance is consistent with its performance during cross-validation:

- MAE: £1,499
- RMSE: £6,836
- $R^2$: 0.908

## 4.4. Model Comparison

In this section, we compare the models proposed in the previous section by reviewing their metrics gathered at the various stages of the modelling phase.

### 4.4.1. 5-Fold Cross-Validation

In [49]:
# Renaming scheme for selected models
selected_models = {
    "Baseline": "Baseline Model",
    "ScaledLR": "Linear Regression",
    "UnconstrainedRF": "Random Forest",
    "FlexibleGB": "Gradient Boosting",
    "SmallNN": "Neural Network"
}

# Combine previous results into one dataset
ungrouped_cv_results = (
    pd.concat([
        baseline_cv_metrics,
        lr_cv_metrics,
        rf_cv_metrics,
        gb_cv_metrics,
        nn_cv_metrics
    ], ignore_index=False)
    .loc[selected_models.keys()]
    .rename(index=selected_models)
)

ungrouped_cv_results

,MAE_Mean,RMSE_Mean,R2_Mean,MAE_SD,RMSE_SD,R2_SD
Baseline Model,10807.132327,22464.410625,-0.046392,31.023583,1020.677465,0.004426
Linear Regression,3575.461536,9595.301929,0.805282,29.577599,2155.388188,0.071121
Random Forest,1006.360090,6629.265674,0.903056,19.539373,2375.958128,0.060829
Gradient Boosting,2580.345963,7634.701655,0.875639,19.371451,2081.037469,0.057378
Neural Network,1521.450384,6432.138337,0.910482,22.433324,1824.828888,0.042592


No singular model has the best metric across the board. Unsurprisingly, the baseline model and linear regression models performed the worst since a constant and a linear model are intrinsicly unable to capture the nonlinear relationships we identified during analysis. Gradient Boosting places third with respect to all metrics. Random Forest has the best MAE with £1,006 but Neural Networks exhibit the best RMSE and $R^2$ score with £6,432 and 0.91 respectively.

Nonetheless, we declare the Random Forest model as the winner. This is because the difference between the RMSE and $R^2$ scores of the Random Forest and Neural Network models are very small:

- RMSE: £6,629 vs. £6,432 (3% larger)
- $R^2$: 0.903 vs. 0.910 (0.8% lower)

But the difference between the MAE is much more significant:

- MAE: £1,006 vs. £1,521 (34% smaller)

Ultimately, deciding between the models results in a trade-off between RMSE and $R^2$, and MAE. Choosing Neural Network would harm the MAE metric much more than choosing Random Forests would harm RMSE and $R^2$. Thus, our ranking of models would be as follows:

1. Random Forests
2. Neural Networks
3. Gradient Boosting
4. Linear Regression

### 4.4.2. Grouped 3-Fold Cross-Validation

In [50]:
# Combine previous results into one dataset
# Note: only collected grouped cv data for selected models, so no need to do selection
grouped_cv_results = (
    pd.concat([
        baseline_cv_grouped_metrics,
        lr_cv_grouped_metrics,
        rf_cv_grouped_metrics,
        gb_cv_grouped_metrics,
        nn_cv_grouped_metrics
    ], ignore_index=False)
    .rename(index=selected_models)
)

grouped_cv_results

,MAE_Mean,RMSE_Mean,R2_Mean,MAE_SD,RMSE_SD,R2_SD
Baseline Model,10807.007892,22473.884399,-0.045987,34.995538,748.399351,0.002859
Linear Regression,3575.461536,9595.301929,0.805282,29.577599,2155.388188,0.071121
Random Forest,1599.965190,5972.839126,0.922826,10.855622,1701.440954,0.039004
Gradient Boosting,2566.111966,6644.834988,0.907453,1.933364,1220.027521,0.028254
Neural Network,1668.356137,7303.829316,0.886344,68.533940,1700.658416,0.041941


The Baseline, Linear Regression, Random Forest and Gradient Boosting models all showed little change when a grouped cross-validation scheme was used. This indicates that the grouping structure in the modelling dataset is not responsible for the performance measured in the previous subsection.
However, Neural Networks demonstrate a much greater sensitivity to the grouped scheme with MAE rising from £1,521 to £1,668, RMSE raising from £6,432 to £7,304 and $R^2$ alling from 0.910 to 0.886.

If this were the primary criterion, then the model ranking would be as follows:

1. Random Forests
2. Gradient Boosting
3. Neural Networks
4. Linear Regression

Notably, the tree-based methods are barely affected by the scheme change and thus Random Forests remain superior.

### 4.4.3. Generalisation

In [51]:
# Combine previous results into one dataset
# Note: these subframes already use the correct name so no need to rename
test_results = pd.concat([
    baseline_errors,
    linear_regression_errors,
    random_forest_errors,
    gradient_boosting_errors,
    neural_network_errors
], ignore_index=False)

All models demonstrated a performance consistent with their performances during the different cross-validation schemes. This indicates that the models' performances were not simply a result of the chosen cross-validation schemes. A performance ranking of the models on the testing set would match that of the 5-fold cross-validation.
As expected from the results of cross-validation, the Random Forest model performed the best out of all models with $R^2$ = 0.924, RMSE = £6,199 and MAE = £902. This provides further support for its selection as the preferred model.

In [52]:
models = {
    "linear_regression.joblib": linear_regression_model,
    "random_forest.joblib": random_forest_model,
    "gradient_boosting.joblib": gradient_boosting_model,
    "neural_network.joblib": neural_network_model
}

for filename, model in models.items():
    joblib.dump(model, f"../models/{filename}")

## 4.5. Model Interpretation

We now take our winning model, the Random Forest model, and attempt to interpret it. Our aim is to evaluate the importance of some of the features of the dataset and attempt to consolidate our findings with our analyses in EDA. We measure the feature importance using permutation importance.

In [53]:
rf_permutation = permutation_importance(
    random_forest_model,
    X_test,
    y_test,
    scoring="neg_mean_absolute_error",
    random_state=42,
    n_jobs=1
)

In [54]:
permutation_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": rf_permutation.importances_mean,
    "SD": rf_permutation.importances_std
}).sort_values("Importance", ascending=False).reset_index(drop=True)

permutation_df

,Feature,Importance,SD
0,EngineBHP,7276.227467,18.611891
1,Year,6256.798039,5.421356
2,Miles,2805.417774,7.926123
3,EngineVol,1581.011305,6.645604
4,Make,1298.900837,3.833921
5,Model,1088.548093,5.149362
6,BodyType,956.037544,2.118545
7,Transmission,943.686705,5.671664
8,FuelType,304.251965,2.714394
9,NumOwner,134.950421,2.319943


`EngineBHP` by far has the largest permutation performance out of any feature but also has a substantially larger variability across these permutations. It's standard deviation is approximately £18.61 compared to Year's £5.42 and Miles' £7.93. Regardless of this, its standard deviation is still relatively close to its mean: £7,276. Also, `Year` is also quite far ahead of `Miles` with an importance of £6,257 vs. £2,805. This is very consistent with the relationships identified during EDA. `Year` was identified as one of the most important predictors since it demonstrated moderate to strong relationships with many of the other features. However, `EngineBHP` demonstrated the weakest monotonic correlation with `Year` with a Spearman coefficient of 0.08 while exhibiting the highest permutation importance.

More surprisingly, `Make` and `Model` exhibit a modest permutation importance compared to `EngineBHP`, `Year` and `Miles`. One possible explanation is that cars from different manufacturers and of different models can share a substantial amount of underlying characteristics. For instance, one manufacturer may own multiple brands (e.g. JLR) and different models produced by different brands could share characteristics with a different model from a different brand under the same manufacturer. Consequently, some of the predictive information that we may have naively attributed to `Make` and `Model` may be better captured by the numerical variables `EngineBHP`, `EngineVol`, `Year` and `Miles` as these best capture the technical characteristics of a vehicle. Nonetheless, this is a speculative interpretation and is not in any way tested by analysis.

`ULEZ` and `BrandNew` have relatively little permutation importance, indicating that they provide limited additional predictive information beyond the other variables in the model.